<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/09-miniproyecto-pronostico-cartera/clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Episodio 1.9 · Miniproyecto: el pronóstico de la cartera

**Serie:** Del Excel al Machine Learning · **Temporada 1** · 🎬 Video próximamente

## 🎯 Qué vas a aprender

- Unir lo aprendido: del archivo sucio al pronóstico y a un Excel.
- Fijar el "hoy" para **comprobar** un pronóstico contra lo que pasó.
- Comparar con reglas simples, antes de creerle a un modelo.
- Presentar el resultado con sus límites.

## 📊 En Excel lo harías así

A fin de noviembre, la gerencia pregunta cuánto se venderá en diciembre, por cartera de vendedor:

1. Limpias la exportación del sistema.
2. Sacas el promedio de los últimos meses.
3. Lo ajustas a ojo porque "diciembre siempre vende más".
4. Armas un Excel.

Y nunca revisas qué tan cerca estuviste. Hoy lo hacemos con un método que se puede comprobar.

## 🐍 En Python

### Paso 1: de lo sucio a lo limpio

Casi todo ya lo conoces, en forma de funciones listas: la limpieza del episodio 0.8, la tabla del 1.4 y `preparar_x` del 1.5. Comprobamos contra la cifra oficial: S/ 13,582,286.61.

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def limpiar_monto(columna):
    return columna.str.replace("S/ ", "").str.replace(",", "").astype(float)


def limpiar_ventas(archivo):
    ventas = pd.read_csv(archivo, sep=";")

    # Nombres de columna
    ventas.columns = ventas.columns.str.lower().str.replace(" ", "_")
    ventas = ventas.rename(
        columns={
            "nro_venta": "id_venta",
            "cliente": "id_cliente",
            "producto": "id_producto",
            "vendedor": "id_vendedor",
            "método_de_pago": "metodo_pago",
        }
    )

    # Duplicados
    ventas = ventas.drop_duplicates()

    # Textos
    ventas["ciudad"] = ventas["ciudad"].str.strip().str.capitalize()
    ventas["metodo_pago"] = ventas["metodo_pago"].str.strip().str.capitalize()
    ventas["metodo_pago"] = ventas["metodo_pago"].replace(
        {"Yape": "Billetera digital", "Plin": "Billetera digital"}
    )
    ventas["metodo_pago"] = ventas["metodo_pago"].fillna("Sin dato")

    # Números guardados como texto
    ventas["precio_unitario"] = limpiar_monto(ventas["precio_unitario"])
    ventas["total"] = limpiar_monto(ventas["total"])
    ventas["descuento"] = ventas["descuento"].str.replace("%", "").astype(float) / 100

    # Fechas
    texto = ventas["fecha"]
    ventas["fecha"] = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
    ventas["fecha"] = ventas["fecha"].fillna(
        pd.to_datetime(texto, format="%Y-%m-%d", errors="coerce")
    )
    ventas["anio"] = ventas["fecha"].dt.year
    ventas["mes"] = ventas["fecha"].dt.month

    # Cantidades vacías o con errores de tipeo
    cantidad = ventas["total"] / (ventas["precio_unitario"] * (1 - ventas["descuento"]))
    ventas["cantidad"] = cantidad.round().astype(int)

    return ventas

In [3]:
def tabla_de_pronostico(ventas, clientes):
    """Una fila por cliente y mes, con cero si no compró, y lo que se sabía antes de ese mes."""
    meses = ventas["fecha"].str[:7].rename("mes")
    todos = pd.MultiIndex.from_product(
        [clientes["id_cliente"], sorted(meses.unique())], names=["id_cliente", "mes"]
    )
    compras = ventas.groupby(["id_cliente", meses])["total"].sum()
    tabla = compras.reindex(todos, fill_value=0).reset_index()

    por_cliente = tabla.groupby("id_cliente")["total"]
    tabla["mes_anterior"] = por_cliente.shift(1)
    tabla["promedio_3"] = por_cliente.transform(lambda s: s.shift(1).rolling(3).mean())
    tabla["promedio_6"] = por_cliente.transform(lambda s: s.shift(1).rolling(6).mean())
    tabla["numero_mes"] = tabla["mes"].str[5:].astype(int)

    columnas = ["id_cliente", "tipo", "ciudad", "id_vendedor", "fecha_alta"]
    tabla = pd.merge(tabla, clientes[columnas], on="id_cliente", how="left")
    tabla = tabla.dropna(subset=["promedio_6"])
    return tabla[tabla["mes"] + "-01" >= tabla["fecha_alta"]].drop(columns="fecha_alta")

In [4]:
def preparar_x(tabla, numeros, textos, columnas=None):
    """Convierte los textos en columnas de 0 y 1. Con `columnas`, usa las de entrenamiento."""
    x = pd.get_dummies(tabla[numeros + textos], columns=textos, dtype=float)
    if columnas is not None:
        x = x.reindex(columns=columnas, fill_value=0)
    return x

In [5]:
ventas = limpiar_ventas(DATOS + "ventas_crudo.csv")
ventas["fecha"] = ventas["fecha"].dt.strftime("%Y-%m-%d")
clientes = pd.read_csv(DATOS + "clientes.csv")

tabla = tabla_de_pronostico(ventas, clientes)

ventas_2024 = ventas[ventas["fecha"].str[:4] == "2024"]
por_mes = ventas_2024.groupby(ventas_2024["fecha"].str[5:7].astype(int))["total"].sum()
indice = por_mes / por_mes.mean()

print(len(ventas), round(ventas["total"].sum(), 2))
print(len(tabla))

50000 13582286.61
5268


### Paso 2: fijamos el "hoy"

> ⚠️ **Estos datos son sintéticos.** Los rangos de error de hoy son de este ejemplo, no de un negocio real.

Hacemos de cuenta que hoy es **30 de noviembre de 2025**. Entrenamos solo con lo que se conocía ese día y pronosticamos diciembre. Como diciembre ya pasó en los datos, podemos comprobarlo. Es una prueba honesta: sin datos del futuro.

> ⚠️ No pronosticamos enero de 2026: no tendríamos cómo verificarlo.

Esta función reúne todo. Entrena la regresión del 1.5, calcula la regla con índice, y suma por cartera:

In [6]:
def pronosticar_cartera(tabla, indice, mes):
    """Entrena con lo anterior a `mes` y compara tres métodos por cartera de vendedor."""
    entreno = tabla[tabla["mes"] < mes]
    objetivo = tabla[tabla["mes"] == mes]

    numeros = ["mes_anterior", "promedio_3"]
    textos = ["tipo", "ciudad", "numero_mes"]
    x_entreno = preparar_x(entreno, numeros, textos)
    x_objetivo = preparar_x(objetivo, numeros, textos, x_entreno.columns)

    modelo = LinearRegression()
    modelo.fit(x_entreno, entreno["total"])

    resultado = objetivo.assign(
        regresion=modelo.predict(x_objetivo),
        regla_indice=objetivo["promedio_6"] * objetivo["numero_mes"].map(indice),
    )
    columnas = ["total", "promedio_6", "regla_indice", "regresion"]
    return resultado.groupby("id_vendedor")[columnas].sum().round(2)

In [7]:
cartera = pronosticar_cartera(tabla, indice, "2025-12")

cartera

,total,promedio_6,regla_indice,regresion
id_vendedor,,,,
E001,65522.75,57757.05,84694.27,88594.24
E002,97802.19,74162.80,108751.48,108971.67
E003,60519.30,44297.48,64957.31,62813.49
E004,75708.15,46841.35,68687.62,71080.04
E005,33330.80,28847.46,42301.59,46727.11
E006,69425.35,50469.77,74008.29,68533.05
E007,57449.46,44861.17,65783.91,63365.03
E008,65570.10,48790.36,71545.62,61789.08
E009,31605.45,22221.55,32585.43,33990.95


### Paso 3: ¿qué tan cerca estuvimos?

In [8]:
print(round(cartera["total"].mean()))
print(round(mean_absolute_error(cartera["total"], cartera["promedio_6"])))
print(round(mean_absolute_error(cartera["total"], cartera["regla_indice"])))
print(round(mean_absolute_error(cartera["total"], cartera["regresion"])))

56751
14256
7316
6090


Una cartera vendió, en promedio, unos S/ 56,751:

- **Promedio de 6 meses:** se equivoca en S/ 14,256. No sabe de la Navidad.
- **Regla con índice:** S/ 7,316.
- **Regresión:** S/ 6,090, cerca del 11 %.

La regresión salió mejor este diciembre, y la regla con índice se equivoca cerca del 13 %. Pero **no declaramos ganador**: repitiendo esto mes a mes de julio a diciembre, cada una gana 3 meses.

En este diciembre, tener en cuenta la temporada redujo el error a la mitad. De julio a diciembre lo redujo un 36 %, y en meses sin pico casi no ayuda.

Y mira el error de la regresión por cartera, en porcentaje:

In [9]:
cartera["error_pct"] = (cartera["regresion"] - cartera["total"]) / cartera["total"] * 100

cartera["error_pct"].round(1).sort_values()

id_vendedor
E012    -6.5
E004    -6.1
E008    -5.8
E013    -5.4
E006    -1.3
E010    -0.8
E014     3.7
E003     3.8
E009     7.5
E007    10.3
E002    11.4
E011    23.2
E001    35.2
E005    40.2
Name: error_pct, dtype: float64

Va del 1 % al 40 %, y con la regla, del 3 % al 29 %. **Por eso esto es un pronóstico de cartera, no una meta.** Si un vendedor quedó por debajo, puede ser por sus clientes, y no por su trabajo. No lo uses para evaluar ni pagar a nadie.

### Paso 4: el Excel

Armamos un archivo con tres hojas, como en el 0.8. Usa códigos de vendedor, y una hoja `Notas` para que nadie lea de más. Los archivos quedan en la carpeta 📁 de Colab:

In [10]:
resumen = pd.DataFrame(
    {
        "metodo": ["Promedio de 6 meses", "Regla con índice", "Regresión"],
        "error_medio_soles": [
            round(mean_absolute_error(cartera["total"], cartera["promedio_6"])),
            round(mean_absolute_error(cartera["total"], cartera["regla_indice"])),
            round(mean_absolute_error(cartera["total"], cartera["regresion"])),
        ],
    }
)

notas = pd.DataFrame(
    {
        "nota": [
            "Diciembre de 2025, un solo mes: no hay un método ganador.",
            "Es un pronóstico de cartera, no una meta.",
            "No debe usarse para evaluar ni pagar a un vendedor.",
            "Datos sintéticos.",
        ]
    }
)
reporte = cartera.reset_index().rename(columns={"total": "venta_real"}).round(1)

with pd.ExcelWriter("pronostico_2025_12.xlsx", engine="openpyxl") as archivo:
    notas.to_excel(archivo, sheet_name="Notas", index=False)
    reporte.to_excel(archivo, sheet_name="Cartera", index=False)
    resumen.to_excel(archivo, sheet_name="Comparación", index=False)

    archivo.sheets["Notas"].column_dimensions["A"].width = 60
    for columna in "ABCDEF":
        archivo.sheets["Cartera"].column_dimensions[columna].width = 14
    for columna in "AB":
        archivo.sheets["Comparación"].column_dimensions[columna].width = 22

pd.read_excel("pronostico_2025_12.xlsx", sheet_name="Comparación")

,metodo,error_medio_soles
0,Promedio de 6 meses,14256
1,Regla con índice,7316
2,Regresión,6090


### Lo que no responde

- **No dice qué producto ofrecer:** los datos no tienen preferencias propias de cada cliente.
- **No sabe por qué:** muestra asociaciones, no causas.
- **Son datos sintéticos.** Sirven para aprender el método, no como conclusión sobre un negocio real.
- **Se apoya en un solo diciembre de historia.** Con el tiempo, habrá que volver a entrenarlo y volver a comprobarlo.

## 🤖 Pídeselo a la IA

> *"Tengo un pronóstico mensual por sucursal con un error medio del 15 %, y hasta 35 % en algunas. Ayúdame a redactar una nota para gerencia que explique para qué sirve, para qué no, y cómo debe leerse."*

La IA te ayuda a escribir y entender código, pero **tú debes entender lo que hace**. Revisa cada respuesta antes de usarla, y no pegues datos de tu empresa.

## ✅ Resumen

| En Excel | En Python |
|---|---|
| Limpiar la exportación | `limpiar_ventas(archivo)` |
| Proyectar con lo que se sabe hoy | `tabla[tabla["mes"] < mes]` |
| Comparar lo proyectado con lo real | `mean_absolute_error(real, pronostico)` |
| Guardar el reporte | `pd.ExcelWriter()` y `.to_excel()` |

## 🏁 Terminaste la Temporada 1

Ya sabes qué es un modelo, cómo se entrena, cómo se comprueba y por qué desconfiar de un buen resultado. Eso es lo que más cuesta en Machine Learning. En la **Temporada 2** veremos más modelos, como árboles de decisión, y hablaremos de ética.

## 📝 Ahora te toca

Abre [el ejercicio en Colab](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/09-miniproyecto-pronostico-cartera/ejercicio.ipynb) y repite el proyecto. Cuando termines, compara con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/09-miniproyecto-pronostico-cartera/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0